# 実際の `erg_lep_part_products` による疑似LEP-iデータの検証

既知の真値を持つH⁺のドリフトMaxwell分布をLEP-i L2形式のtplot変数に格納し、**インストール済みの公開関数を実際に呼び出す**。関数・getter・moment計算を置換するmockやmonkeypatchは使わない。

既存の `LEPi_drifting_Maxwellian_partial_moments.ipynb` は連続積分の基準として残す。今回の検証では次の三つを区別する。

1. **全分布の真値**：指定した密度 $n$、速度 $\mathbf{U}$、温度 $\theta=k_{\mathrm{B}}T$、圧力 $p=n\theta$。
2. **連続積分のpartial moment**：球殻 $30\,\mathrm{eV}\le E\le30\,\mathrm{keV}$ 内での値。未観測領域を除くことによる偏りを表す。
3. **実関数の出力**：ネイティブビンでサンプルした疑似fluxを公開関数に渡した値。2との差にはビン選択・ビン幅・サンプリング・角度積分の影響が含まれる。

さらに、3を**同じ入力値・同じビンに対する独立な離散計算**と比較する。角度積分は独立なGauss–Legendre積分で求め、ライブラリの角度重み関数は使わない。これにより、離散化の影響と、与えられた配列に対する処理の整合性を分ける。

実観測flux、ネットワーク取得、長期間の解析は使わない。ローカルCDFからコピーしたエネルギー・角度設定だけを `lepi_l2_geometry.json` に保存している。全セルを上から実行し、結果は `outputs/` に保存する。


## 入力と単位

疑似分布は

$$
f_{\mathrm{SI}}(\mathbf{v})=\frac{n_{\mathrm{SI}}}{\pi^{3/2}v_{\mathrm{th}}^3}
\exp\!\left[-\frac{|\mathbf{v}-\mathbf{U}|^2}{v_{\mathrm{th}}^2}\right],
\qquad v_{\mathrm{th}}=\sqrt{\frac{2\theta_{\mathrm{J}}}{m_{\mathrm{SI}}}}
$$

とする。速度軸はgetterが生成する粒子の進行方向のCartesian軸 $(x,y,z)$ であり、磁場に対する平行・垂直軸ではない。エネルギー選択と流速の指定は同じ座標系で行う。

分布関数から差分数fluxを**ライブラリの単位変換を使わず**に作る。$\epsilon_{\mathrm{eV}}=1.602176634\times10^{-19}\,\mathrm{J/eV}$ とすると、

$$
j_{\mathrm{eV}}(E,\hat{\mathbf{v}})
=f_{\mathrm{SI}}(v\hat{\mathbf{v}})\,v^2\frac{\epsilon_{\mathrm{eV}}}{m_{\mathrm{SI}}}\times10^{-4}
$$

は $\mathrm{cm}^{-2}\,\mathrm{s}^{-1}\,\mathrm{sr}^{-1}\,\mathrm{eV}^{-1}$ 単位になる。L2入力にはこれを1000倍し、keVあたりのfluxとして格納する。

入力配列は `[time, 30 energy, 8 anode, 16 sector]`。ローダーと同じくセクタ座標を $0,\ldots,15$ とする。CDFのエネルギー設定は最初の30チャンネルを使い、getterが除外するチャンネル0も入力に残す。

まずエネルギー・角度中心でのflux値を使い、後半でビン平均値との差を調べる。どちらも校正済み検出器応答の再現ではない。離散化の検証で物理定数の差を混ぜないため、疑似分布と連続基準ではgetterの質量定数を使う。そのSI陽子質量との差も記録する。

公開関数への指定は `outputs=['moments']`, `energy=[30, 30000]`、その他の角度制限は既定値とする。磁場には補助的な定数ベクトルを与え、FAC回転は要求しない。衛星電位補正もこの公開経路では与えない。


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.constants import proton_mass, elementary_charge
import lepi_synthetic_validation as validation

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
PRIMARY_PACKAGE = 'pyspedas'  # 現行の研究ノートは psp.projects.erg... を呼ぶ
REFERENCE_DENSITY_CM3 = 1.0
TEMPERATURES_EV = np.array([1., 3., 10., 30., 100., 300., 1000., 3000., 10000.])
SPEEDS_KMS = np.array([0., 10., 30., 50., 100., 200., 300.])
DIRECTIONS = {'x': np.array([1., 0., 0.]),
              'y': np.array([0., 1., 0.]),
              'z': np.array([0., 0., 1.])}
print('Python:', sys.version.split()[0], platform.platform())
print('Primary implementation:', PRIMARY_PACKAGE)


## 実行環境と使用したソース

研究ノートが呼ぶ `pyspedas` 版を主対象にし、PIのコメントで言及された `ergpyspedas` 版との比較も行う。インストール済みソースのパスとSHA-256を記録する。結果はこの環境・この実装に対するものであり、将来の版に無条件には適用しない。

処理中のruntime warningとlogging warningを捕捉し、件数とloggingの内容をCSVに記録する。たとえば `ergpyspedas` 版の出力表示設定が存在しない変数を参照するメッセージも、数値計算の照合結果と区別して残す。補助コードは [lepi_synthetic_validation.py](lepi_synthetic_validation.py) にある。

In [ ]:
manifest = validation.source_manifest()
manifest['python'] = sys.version
for name in ['lepi_synthetic_validation.py', 'lepi_l2_geometry.json',
             'LEPi_drifting_Maxwellian_partial_moments.ipynb']:
    manifest[name+'_sha256'] = hashlib.sha256(Path(name).read_bytes()).hexdigest()
(OUTPUT_DIR/'public_validation_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+'\n')
display(pd.DataFrame([manifest['packages']]))
print('Geometry source:', manifest['geometry']['source_cdf'])
native = validation.native_distribution(PRIMARY_PACKAGE)
mass_si = native['mass']*elementary_charge/1e6
print('Getter mass [eV/(km/s)^2]:', native['mass'])
print('Getter mass / scipy proton mass - 1:', mass_si/proton_mass-1)


## ビンと積分範囲の確認

エネルギー範囲の指定はビン中心値で採否を決める。ビンの端を30 eVと30 keVで切り直す処理ではない。getterから得た幅と、その実装で定義される境界を表示する。`getter_lower_edge_ev` / `getter_upper_edge_ev` はその幅の由来を確認するための値で、中心値 $\pm\Delta E/2$ と一致するとは限らない。

角度中心がCDFのSGA設定と対応すること、各エネルギーの角度ビンの立体角総和が $4\pi$、方向ベクトルの積分がゼロ、dyadの積分が $(4\pi/3)\mathbf{I}$ になることも独立に確認する。ただし、実際の衛星姿勢や座標変換の正しさはこの疑似データ検証の対象外である。


In [ ]:
bin_table = pd.DataFrame(validation.native_bin_table(PRIMARY_PACKAGE))
bin_table.to_csv(OUTPUT_DIR/'native_energy_bins.csv', index=False)
display(bin_table)
selected = bin_table.loc[bin_table.selected]
support = [selected.getter_lower_edge_ev.min(), selected.getter_upper_edge_ev.max()]
print('Selected centre range [eV]:', selected.energy_ev.min(), selected.energy_ev.max())
print('Selected bin-support range [eV]:', support)
root = ('pyspedas.projects.erg' if PRIMARY_PACKAGE == 'pyspedas' else 'ergpyspedas.erg')
import importlib
angle_fn = importlib.import_module(root+'.satellite.erg.particle.get_lepi_flux_angle_in_sga').get_lepi_flux_angle_in_sga
getter_angles = angle_fn()[:, 1, :8]
cdf_angles = np.array(validation.GEOMETRY['angle_centres_sga_degrees'])
np.testing.assert_allclose(getter_angles[0], cdf_angles[0], atol=1e-10)
# Azimuths -90 and 270 degrees denote the same direction.
np.testing.assert_allclose((getter_angles[1]-cdf_angles[1]+180)%360-180, 0., atol=1e-10)
omega, angular_vector, angular_tensor = validation.independent_angular_weights(PRIMARY_PACKAGE)
print('Solid-angle and angular-tensor checks passed.')
fig, ax = plt.subplots(figsize=(9, 3), constrained_layout=True)
ax.errorbar(selected.energy_ev, np.zeros(len(selected)),
            xerr=[selected.energy_ev-selected.getter_lower_edge_ev,
                  selected.getter_upper_edge_ev-selected.energy_ev], fmt='o', markersize=4)
for edge in validation.ENERGY_RANGE:
    ax.axvline(edge, color='0.5', ls='--')
ax.set_xscale('log')
ax.set_xlabel(r'Energy $E$ [$\mathrm{eV}$]')
ax.set_yticks([])
ax.set_title('Selected LEP-i bins and requested limits')
fig.savefig(OUTPUT_DIR/'native_energy_bins.png', dpi=180)
plt.show()


## 公開関数・独立離散計算・連続基準の比較

独立離散計算では、各ビンの $j(E_c)$、$\Delta E$、速度 $v(E_c)$ を用いて

$$
n_{\mathrm{disc}}=\sum j(E_c)\frac{\Delta E}{v(E_c)}\int_{\mathrm{bin}}\mathrm{d}\Omega,
\quad
\mathbf{F}_{\mathrm{disc}}=\sum j(E_c)\Delta E\int_{\mathrm{bin}}\hat{\mathbf{v}}\,\mathrm{d}\Omega
$$

を計算する。式では速度・flux・密度の長さ単位を揃える。2次momentも $\hat{\mathbf{v}}\hat{\mathbf{v}}$ の独立な角度積分から求め、partial速度で中心化する。角度積分と、fluxからmomentへの計算は公開関数の実装を呼ばない。

密度・ベクトルflux・速度・圧力テンソル・温度テンソル・平均温度を照合する。ゼロに近いベクトル成分を含むため、比較の絶対許容誤差は各量の代表スケールから定める（$2\times10^{-10}$倍）。fluxには $n v_{\mathrm{th}}$ のスケールも用いる。

圧力は公開関数の `ptens`（eV/cm³）、温度は `avgtemp`（eV）を使う。Paへの変換は $1\,\mathrm{eV/cm^3}=1.602176634\times10^{-13}\,\mathrm{Pa}$。


In [ ]:
comparison_rows = []
for package in ['pyspedas', 'ergpyspedas']:
    for theta, speed, direction in [(10., 20., 'x'), (30., 30., 'z'),
                                     (100., 50., 'y'), (1000., 0., 'x'),
                                     (1000., 100., 'x'), (1000., 100., 'z'),
                                     (10000., 100., 'x')]:
        r = validation.result_record(theta, speed*DIRECTIONS[direction],
                                     REFERENCE_DENSITY_CM3, package)
        comparison_rows.append(dict(direction=direction, **r))
comparisons = pd.DataFrame(comparison_rows)
comparisons.to_csv(OUTPUT_DIR/'public_implementation_comparison.csv', index=False)
fields = ['density_ratio', 'projected_speed_kms', 'temperature_ratio', 'pressure_ratio']
for field in fields:
    a = comparisons[comparisons.package == 'pyspedas'][field].to_numpy()
    b = comparisons[comparisons.package == 'ergpyspedas'][field].to_numpy()
    np.testing.assert_allclose(a, b, rtol=1e-12, atol=1e-10)
display(comparisons[['package', 'theta_ev', 'true_speed_kms', 'direction',
                     'density_ratio', 'projected_speed_kms', 'temperature_ratio',
                     'pressure_ratio', 'max_discrete_scaled_error']])
print('Both public implementations agree for the tested cases.')
if comparisons.public_logging_warning_count.sum():
    display(comparisons.loc[comparisons.public_logging_warning_count > 0,
                            ['package', 'public_logging_messages']].drop_duplicates())


## ネイティブビンでの温度・流速・方向の走査

$x$、$y$、$z$ 各軸のドリフトを走査する。すべての点で公開関数を実行し、独立離散計算との照合を行う。速度は入力ドリフト方向への射影 $\mathbf{U}_{\mathrm{out}}\cdot\hat{\mathbf{U}}$ と、それに垂直な成分を記録する。$U=0$ の速度比は定義しない。回収密度が指定真値の $10^{-10}$ 以下の場合、比を解釈しない領域として図でマスクする。

温度・流速範囲は例示であり、実際の対象領域の分布を拘束するものではない。

In [ ]:
scan_rows = []
for direction, nhat in DIRECTIONS.items():
    for theta in TEMPERATURES_EV:
        for speed in SPEEDS_KMS:
            r = validation.result_record(theta, speed*nhat, REFERENCE_DENSITY_CM3,
                                         PRIMARY_PACKAGE)
            scan_rows.append(dict(direction=direction, **r))
    print('Completed direction:', direction)
scan = pd.DataFrame(scan_rows)
scan.to_csv(OUTPUT_DIR/'public_parameter_scan.csv', index=False)
print('Public calls:', len(scan))
print('Maximum scaled difference from independent discrete reference:', scan.max_discrete_scaled_error.max())
print('Captured public runtime warnings:', scan.public_warning_count.sum())
print('Captured public logging warnings:', scan.public_logging_warning_count.sum())


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12), constrained_layout=True)
fields = [('density_ratio', r'$n_{\mathrm{out}}/n$'),
          ('velocity_ratio', r'$(\mathbf{U}_{\mathrm{out}}\cdot\hat{\mathbf{U}})/U$'),
          ('temperature_ratio', r'$\theta_{\mathrm{out}}/\theta$'),
          ('pressure_ratio', r'$p_{\mathrm{out}}/p$')]
for row, direction in enumerate(DIRECTIONS):
    subset = scan[scan.direction == direction]
    for col, (field, title) in enumerate(fields):
        ax = axes[row, col]
        grid = subset.pivot(index='theta_ev', columns='true_speed_kms', values=field)
        values = grid.to_numpy().copy()
        # Include density recovery, but mask thermal/speed ratios with too little density.
        if field != 'density_ratio':
            density_grid = subset.pivot(index='theta_ev', columns='true_speed_kms', values='density_ratio')
            values[density_grid.to_numpy() <= validation.DENSITY_FLOOR] = np.nan
        cmap = plt.get_cmap('viridis').with_extremes(bad='0.8')
        global_values = scan[field].to_numpy()
        finite = global_values[np.isfinite(global_values)]
        im = ax.pcolormesh(grid.columns, grid.index, values, shading='nearest', cmap=cmap,
                           vmin=finite.min(), vmax=finite.max())
        ax.set_yscale('log')
        ax.set_xlabel(r'True $U$ [$\mathrm{km\,s^{-1}}$]')
        ax.set_ylabel(r'True $\theta$ [$\mathrm{eV}$]')
        ax.set_title(direction+' drift: '+title)
        fig.colorbar(im, ax=ax)
fig.suptitle(PRIMARY_PACKAGE+': actual public function on native LEP-i bins')
fig.savefig(OUTPUT_DIR/'public_parameter_scan.png', dpi=160)
plt.show()


## 1 keV分布：連続積分から実装出力まで

公開関数、要求した球殻の連続積分、実際に選択されたビンの外側境界での連続積分を比較する。外側境界の基準は範囲選択の影響を区別するためのものであり、公開関数がビン内の分布を連続積分していることを意味しない。

公開関数の出力圧力が真値をわずかに上回る場合がある。連続的な制限積分に対する上限の性質を、中心サンプルと有限幅を使う数値積分にそのまま課すことはできない。


In [ ]:
one_kev = scan[scan.theta_ev == 1000.].copy()
display(one_kev[['direction', 'true_speed_kms', 'density_ratio', 'velocity_ratio',
                 'temperature_ratio', 'pressure_ratio', 'off_axis_speed_kms']])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for direction in DIRECTIONS:
    sub = one_kev[one_kev.direction == direction]
    axes[0].plot(sub.true_speed_kms, sub.projected_speed_kms, 'o-', label=direction+' drift: public')
    axes[1].plot(sub.true_speed_kms, sub.temperature_ratio, 'o-', label=direction+' drift: public')
    axes[2].plot(sub.true_speed_kms, sub.pressure_ratio, 'o-', label=direction+' drift: public')
ref = one_kev[one_kev.direction == 'x']
for ax, ideal, shell in zip(axes,
                          ['continuous_speed_kms', 'continuous_temperature_ratio', 'continuous_pressure_ratio'],
                          ['shell_speed_kms', 'shell_temperature_ratio', 'shell_pressure_ratio']):
    ax.plot(ref.true_speed_kms, ref[ideal], 'k--', label='Requested shell: continuous')
    ax.plot(ref.true_speed_kms, ref[shell], color='0.5', ls=':', label='Bin support: continuous')
    ax.set_xlabel(r'True $U$ [$\mathrm{km\,s^{-1}}$]')
    ax.grid(alpha=.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel(r'Projected $U_{\mathrm{out}}$ [$\mathrm{km\,s^{-1}}$]')
axes[1].set_ylabel(r'$\theta_{\mathrm{out}}/\theta$')
axes[2].set_ylabel(r'$p_{\mathrm{out}}/p$')
fig.savefig(OUTPUT_DIR/'public_one_kev.png', dpi=180)
plt.show()
print('1 keV ratio ranges:')
display(one_kev[['density_ratio', 'velocity_ratio', 'temperature_ratio', 'pressure_ratio']].agg(['min', 'max']))


## ビン内のサンプリングへの感度

中心でのflux値と、矩形のエネルギー・角度ビン内で平均したflux値を比較する。ビン平均は一様なtop-hat応答を仮定した例であり、LEP-iの校正済み応答を再現していない。両方の入力について公開関数と独立離散計算の一致を確認する。

ビン平均生成のGauss積分は8点と16点で収束を確認する（1 keV、100 km/sの例）。中心サンプルに固有の結果を、検出器の一般的な誤差として扱わない。

In [ ]:
sampling_rows = []
for theta, speed in [(30., 30.), (1000., 100.)]:
    for direction in DIRECTIONS:
        for sampling in ['centre', 'bin_average']:
            r = validation.result_record(theta, speed*DIRECTIONS[direction],
                                         REFERENCE_DENSITY_CM3, PRIMARY_PACKAGE, sampling)
            sampling_rows.append(dict(direction=direction, **r))
sampling_results = pd.DataFrame(sampling_rows)
sampling_results.to_csv(OUTPUT_DIR/'public_sampling_sensitivity.csv', index=False)
f8 = validation.make_flux(1000., [100.,0.,0.], sampling='bin_average', quadrature_order=8)
f16 = validation.make_flux(1000., [100.,0.,0.], sampling='bin_average', quadrature_order=16)
np.testing.assert_allclose(f8, f16, rtol=1e-8, atol=1e-10)
display(sampling_results[['theta_ev', 'true_speed_kms', 'direction', 'sampling',
                          'density_ratio', 'velocity_ratio', 'temperature_ratio', 'pressure_ratio']])
print('Bin-average quadrature convergence check passed.')


## 積分コアの解像度収束

LEP-i getterは30エネルギーチャンネルを前提としているため、その公開経路に任意の細分ビンを渡すことはしない。ここでは**診断として下流の実際の積分コアだけ**に、細分化した理想的な全立体角ビンを与える。同じcleaning・単位変換・moment関数を呼び、要求した球殻の連続積分へ収束するかを調べる。

この試験はネイティブLEP-iの解像度が改善されることを意味しない。公開関数そのものの試験は前のセルで行っている。

In [ ]:
theta, u = 1000., np.array([100.,0.,0.])
continuous = validation.continuous_reference(theta, np.linalg.norm(u), package=PRIMARY_PACKAGE)
refinement_rows = []
for ne, nphi, ntheta in [(24,16,8), (48,32,16), (96,64,32)]:
    out = validation.refined_core_moments(theta, u, ne, nphi, ntheta, package=PRIMARY_PACKAGE)
    p = np.sum(out['ptens'][:3])/3
    refinement_rows.append(dict(ne=ne, nphi=nphi, ntheta=ntheta,
                                 density_ratio=out['density'], speed_kms=out['velocity'][0],
                                 temperature_ratio=out['avgtemp']/theta, pressure_ratio=p/theta,
                                 density_error=out['density']-continuous['density_fraction'],
                                 speed_error_kms=out['velocity'][0]-continuous['partial_speed_kms'],
                                 temperature_error=out['avgtemp']/theta-continuous['temperature_ratio'],
                                 pressure_error=p/theta-continuous['pressure_ratio']))
refinement = pd.DataFrame(refinement_rows)
for field in ['density_error', 'speed_error_kms', 'temperature_error', 'pressure_error']:
    errors = np.abs(refinement[field].to_numpy())
    assert np.all(np.diff(errors) < 0), field
refinement.to_csv(OUTPUT_DIR/'core_refinement.csv', index=False)
display(refinement)
print('Core-grid refinement converges toward the continuous reference.')


## 無効値・負値・スケーリングの確認

NaN・Infを含むビンは除外され、負fluxはgetterでゼロにされるという現在の処理を、独立離散計算と照合する。欠測成分の物理的な影響を補正する試験ではない。

また密度の入力を変えたとき、速度・温度が変わらず、密度・圧力が比例することを確認する。

In [ ]:
baseline_flux = validation.make_flux(1000., [100.,0.,0.])
invalid_rows = []
for label, value in [('NaN', np.nan), ('Inf', np.inf), ('negative', -1.)]:
    flux = baseline_flux.copy()
    flux[10:13, 2:4, 3:6] = value
    for package in ['pyspedas', 'ergpyspedas']:
        public = validation.run_public(flux, package)
        independent = validation.independent_discrete_moments(flux, package)
        error = validation.check_public_vs_discrete(public, independent, package)
        invalid_rows.append(dict(case=label, package=package, density=public['density'],
                                 avgtemp_ev=public['avgtemp'], max_discrete_scaled_error=error))
invalid_checks = pd.DataFrame(invalid_rows)
invalid_checks.to_csv(OUTPUT_DIR/'public_invalid_value_checks.csv', index=False)
display(invalid_checks)
r1 = validation.result_record(1000., [100.,0.,0.], density_cm3=1.)
r2 = validation.result_record(1000., [100.,0.,0.], density_cm3=.01)
for field in ['projected_speed_kms', 'temperature_ev']:
    np.testing.assert_allclose(r1[field], r2[field], rtol=1e-12, atol=1e-10)
for field in ['density_cm3', 'pressure_pa']:
    np.testing.assert_allclose(r2[field], .01*r1[field], rtol=1e-12, atol=1e-20)
print('Invalid-value handling and density scaling checks passed.')
validation.clear_synthetic()


## この検証から言えること・言えないこと

- 同じ疑似flux配列とビンに対する独立計算との一致は、試した入力に対する密度・速度・温度・圧力の処理の整合性を示す。すべての入力・バージョンの正しさを保証するものではない。
- 連続基準との差は、エネルギー中心値による採否、有限ビンでのエネルギー積分、角度内での分布の表現を含む。単一の原因へ一括して帰属させない。
- 同じ温度でもドリフト方向で結果が変わり得る。ネイティブ角度ビンの影響を含むため、既存の連続積分だけから実関数の精度を判断しない。
- 今回はH⁺、定常・等方的な単一Maxwell分布、全角度、有効な正のエネルギーを中心に試験している。He⁺・O⁺の合成moment、非Maxwell分布、時間変動、FAC出力、衛星姿勢、実際の検出効率・応答、背景・計数統計、衛星電位による分布の変形は未検証である。
- 入力方向をgetterの角度中心に合わせて疑似データを作るため、実観測の粒子方向・座標系の絶対的な正しさをこの比較だけでは保証できない。CDFのSGA角度との一致は設定の確認に留まる。
- 観測されたpartial温度が1 keVでも、未観測の冷たい成分が小さいとは保証されない。関数の処理の整合性と、実観測から全分布のmomentを推定できるかは別の問いである。

PIへの報告では「連続積分の基準」「実公開関数への疑似データ入力」「独立離散計算との一致」「分布・応答の仮定と未検証の範囲」を分けて記載する。
